# 🧠 NeuralDBG v1.5.0 — Causal Debugging Demo

**Detect WHY your PyTorch model failed — in 2 minutes, on CPU.**

This notebook demonstrates the core features:
1. **Hook-based monitoring** — wrap any model with `NeuralDbg()`
2. **Causal chains** — trace root cause → symptom
3. **Vanishing gradient detection** — catch silent failures
4. **Auto-fix suggestions** — actionable remediation

✅ Works on **CPU** (no GPU needed) • ✅ **Free** Colab tier

In [ ]:
# @title Install NeuralDBG (run once)
!pip install -q "neuraldbg>=1.3.1" torch

import torch
import torch.nn as nn
from neuraldbg import NeuralDbg

print(f"✅ NeuralDBG + PyTorch {torch.__version__} ready!")

In [ ]:
# @title 1. Build a model with a HIDDEN bug (Sigmoid = vanishing risk)
import torch.nn.functional as F

class BuggyModel(nn.Module):
    """A simple MLP that WILL fail due to Sigmoid saturation."""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 10)
        # 🐛 BUG: Sigmoid saturates -> gradients vanish
        self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        return self.fc3(x)

model = BuggyModel()
print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")
print("⚠️  Using Sigmoid — will cause vanishing gradients!")

In [ ]:
# @title 2. Train with NeuralDBG monitoring
from neuraldbg import NeuralDbg

# Wrap model — this installs forward/backward hooks
with NeuralDbg(model) as dbg:
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()
    
    losses = []
    for step in range(20):
        x = torch.randn(8, 16)
        y = torch.randint(0, 10, (8,))
        
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        dbg.step_iteration()
        dbg.record_loss(loss.item())
        opt.step()
        losses.append(loss.item())
    
    # Get diagnosis
    events = dbg.dump_events()
    hypotheses = dbg.explain_failure()
    chains = dbg.explain_causal()

print(f"📊 Events captured: {len(events)}")
print(f"🔗 Causal chains:   {len(chains)}")
print(f"💡 Hypotheses:      {len(hypotheses)}")

In [ ]:
# @title 3. Show vanishing events and causal chains
vanishing = [e for e in events 
             if 'vanishing' in str(e.get('to_state', '')).lower()]

print(f"🔴 Vanishing gradient events: {len(vanishing)}")
for v in vanishing[:3]:
    print(f"   Layer: {v['layer_name']:20s} | {v['from_state']} -> {v['to_state']}")

print(f"\n🔗 Top causal chain:")
if chains:
    chain = chains[0]
    print(f"   {chain.root_cause}  →  {chain.final_symptom}")

print(f"\n💡 Best hypothesis:")
if hypotheses:
    h = hypotheses[0]
    print(f"   {h.description[:120]}...")
    print(f"   Confidence: {h.confidence:.0%}")

## 🩹 4. The Fix: Replace Sigmoid with ReLU

NeuralDBG told us the root cause: **Sigmoid saturation causing vanishing gradients**.

The fix is simple — swap `Sigmoid()` for `ReLU()` and retrain:

In [ ]:
# @title Fixed model: ReLU instead of Sigmoid
class FixedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 10)
        self.activation = nn.ReLU()  # ✅ FIXED

    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        return self.fc3(x)

fixed = FixedModel()

with NeuralDbg(fixed) as dbg:
    opt = torch.optim.SGD(fixed.parameters(), lr=0.01)
    for step in range(20):
        x = torch.randn(8, 16); y = torch.randint(0, 10, (8,))
        opt.zero_grad()
        loss = nn.CrossEntropyLoss()(fixed(x), y)
        loss.backward()
        dbg.step_iteration()
        dbg.record_loss(loss.item())
        opt.step()
    
    fixed_events = dbg.dump_events()
    fixed_vanishing = [e for e in fixed_events 
                       if 'vanishing' in str(e.get('to_state', '')).lower()]

print(f"🔴 Before fix: {len(vanishing)} vanishing events")
print(f"🟢 After fix:  {len(fixed_vanishing)} vanishing events")
if len(fixed_vanishing) == 0:
    print("✅ Vanishing gradients ELIMINATED!")

## 🔬 Try it on YOUR model

Replace `YourModel` below with any PyTorch model:

```python
from neuraldbg import NeuralDbg

model = YourModel()  # <-- your model here

with NeuralDbg(model) as dbg:
    # ... your training loop ...
    dbg.step_iteration()
    dbg.record_loss(loss.item())

# After training:
events = dbg.dump_events()
chains = dbg.explain_causal()
hypotheses = dbg.explain_failure()

for h in hypotheses:
    print(f"{h.description} (confidence: {h.confidence:.0%})")
```

**NeuralDBG detects**: vanishing/exploding gradients, NaN, dead neurons, optimizer instability, data anomalies — across MLP, CNN, RNN, Transformer, GNN, MoE, Diffusion, and more.

📦 `pip install neuraldbg` • ⭐ [GitHub](https://github.com/LambdaSection/NeuralDBG)